# 04 Reporting and decision summary

Notebook 04 completes the GA4 capstone pipeline by turning the final diagnostics and model outputs into reporting artefacts. It reads the final dashboard/reporting CSVs from the Power BI latest folder and prepares a dynamic decision summary that Power BI can use for weekly issue cards, trend labels, model evidence, and intervention review messaging.

## Purpose

- Load the final funnel, Top Issues, model, feature importance, and intervention review CSV outputs from `powerbi_extracts/latest`
- Build a dynamic `weekly_decision_summary.csv` for Power BI rather than hardcoding dashboard text
- Add issue trend fields, dashboard evidence text, action categories, and suggested owner fields
- Add model evidence and intervention group rows to the same decision summary table
- Save a deterministic weekly memo using the same evidence as the dashboard
- Save a compact audit JSON for reproducibility and final report checks

## Process

1. Set up the reporting folders, fixed project date window, run timestamp, and lightweight helper functions used by the final reporting layer.
2. Load the stable outputs from Notebooks 02 and 03 from `powerbi_extracts/latest`, including Top Issues, model metrics, grouped feature importance, and combined intervention candidates.
3. Summarise the latest Top Issues, final model evidence, ranking cut-offs, feature groups, and intervention review groups.
4. Build a Power BI decision summary that includes Top Issue rows, model evidence rows, and intervention group rows.
5. Add dynamic text fields, trend/status fields, funnel ownership fields, and visual sort helpers so Power BI pages can refresh without manual text edits.
6. Save the weekly decision summary, deterministic memo, and compact audit for the final reporting stage.


# 1.0 Data imports and environment setup


In [1]:
#------------------------------------------------------------------------------
# 1.1 Import libraries and mount Drive
#------------------------------------------------------------------------------
from pathlib import Path
from datetime import datetime, timezone
import json, time
import numpy as np, pandas as pd
from google.colab import drive
from IPython.display import display, Markdown

# Mount Drive for project files
drive.mount("/content/drive", force_remount=True)

# Store run timing values
NOTEBOOK_START_TS = time.time()
RUN_TS = datetime.now(timezone.utc)
RUN_DATE_TAG = RUN_TS.strftime("%Y%m%d")

# Set wider table display options
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


Mounted at /content/drive


# 2.0 Load reporting inputs and helper functions

Notebook 04 reads the final dashboard/reporting CSV outputs rather than recreating diagnostics or modelling logic. The Power BI latest folder is used as the single source of truth for final CSV tables, while the audit and memo are saved separately as reporting artefacts.

**Process**:
1. Define the shared capstone folders, fixed date window, and final input/output paths used by the reporting layer.
2. Define compact helpers for file loading, CSV/JSON saving, percentage formatting, funnel-step labels, and action text.
3. Load the final funnel, Top Issues, model, ranking, feature importance, and combined intervention candidate CSV tables from `powerbi_extracts/latest`.
4. Confirm the required reporting columns exist, including the final Notebook 03 intervention score fields.
5. Prepare clean reporting labels and score-field names used by the later summaries.


In [2]:
#------------------------------------------------------------------------------
# 2.1 Core config and final file paths
#------------------------------------------------------------------------------
# Set core project folders
CAPSTONE_DIR = Path("/content/drive/MyDrive/Capstone_Project")
OUTPUT_DIR = CAPSTONE_DIR / "outputs"
DQ_DIR = OUTPUT_DIR / "data_quality"
REPORTS_DIR = CAPSTONE_DIR / "reports"
PBI_LATEST_DIR = CAPSTONE_DIR / "powerbi_extracts" / "latest"

# Create output folders
for path in [REPORTS_DIR, PBI_LATEST_DIR, DQ_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Set fixed GA4 table date window
START_DATE = "20201101"
END_DATE = "20210228"

# Set final input paths
FUNNEL_WEEKLY_PATH = PBI_LATEST_DIR / "funnel_weekly_rates.csv"
SEGMENT_STEP_PATH = PBI_LATEST_DIR / "segment_step_metrics_weekly.csv"
TOP_ISSUES_PATH = PBI_LATEST_DIR / "top_issues_weekly.csv"
MODEL_COMPARISON_PATH = PBI_LATEST_DIR / "model_comparison_metrics.csv"
RANKING_CUTOFF_PATH = PBI_LATEST_DIR / "ranking_cutoff_metrics.csv"
GROUPED_IMPORTANCE_PATH = PBI_LATEST_DIR / "grouped_model_importance.csv"
COMBINED_CANDIDATES_PATH = PBI_LATEST_DIR / "combined_priority_candidates.csv"

# Set final output paths
WEEKLY_DECISION_SUMMARY_PATH = PBI_LATEST_DIR / "weekly_decision_summary.csv"
WEEKLY_MEMO_PATH = REPORTS_DIR / f"weekly_memo_template_{RUN_DATE_TAG}.md"
NOTEBOOK04_AUDIT_PATH = DQ_DIR / "notebook04_audit.json"

print("CAPSTONE_DIR:", CAPSTONE_DIR)
print("PBI_LATEST_DIR:", PBI_LATEST_DIR)
print("Project date window:", START_DATE, "to", END_DATE)
print("RUN_DATE_TAG:", RUN_DATE_TAG)


CAPSTONE_DIR: /content/drive/MyDrive/Capstone_Project
PBI_LATEST_DIR: /content/drive/MyDrive/Capstone_Project/powerbi_extracts/latest
Project date window: 20201101 to 20210228
RUN_DATE_TAG: 20260607


In [3]:
#------------------------------------------------------------------------------
# 2.2 Compact helper functions
#------------------------------------------------------------------------------
# Print a simple section banner
def print_banner(title):
    print("\n" + "-" * 78); print(title); print("-" * 78)

# Read a required CSV
def read_csv_required(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

# Check required columns
def require_cols(df, cols, df_name):
    missing = [col for col in cols if col not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing required columns: {missing}")

# Save JSON with overwrite
def save_json_overwrite(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

# Save CSV with parent folder creation
def save_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

# Format percentages for reporting text
def pct_text(x, d=1):
    return "not available" if pd.isna(x) else f"{x:.{d}%}"

# Format numbers for reporting text
def num_text(x, d=1):
    return "not available" if pd.isna(x) else f"{x:,.{d}f}"

# Standardise weekly date columns
def clean_date_col(df, col="week_start_date"):
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce").dt.date.astype("string")
    return df

# Convert funnel step names into readable labels
def step_display(step):
    step_map = {"view_item":"product view", "add_to_cart":"cart", "begin_checkout":"checkout", "add_shipping_info":"shipping", "add_payment_info":"payment", "purchase":"purchase"}
    return step_map.get(str(step), str(step).replace("_", " "))

# Build a stable key for comparing issues over time
def build_issue_key(row):
    fields = [row.get("segment_type", ""), row.get("segment_value", ""), row.get("step_from", ""), row.get("step_to", "")]
    return " | ".join([str(x) for x in fields])

# Convert raw segment values into readable labels
def clean_display_label(row):
    raw_label = row.get("segment_value_display", None)
    raw_value = row.get("segment_value", "this segment")
    label = raw_value if pd.isna(raw_label) or str(raw_label).strip() == "" else raw_label
    label = str(label).strip()
    label_lower = label.lower()

    label_map = {
        "google / organic": "Google organic traffic",
        "google / cpc": "Google paid search traffic",
        "(direct) / (none)": "Direct traffic",
        "(direct) / organic": "Direct organic traffic",
        "(direct) / referral": "Direct referral traffic",
        "shop.googlemerchandisestore.com / referral": "Google Merch Store referral traffic",
        "analytics.google.com / referral": "Google Analytics referral traffic",
        "<other> / <other>": "Other channels",
        "<other> / referral": "Other referral traffic",
        "<other> / organic": "Other organic traffic",
        "(data deleted) / (data deleted)": "Unknown traffic",
        "(data deleted) / organic": "Unknown organic traffic",
        "(data deleted) / referral": "Unknown referral traffic",
        "(not set) / (not set)": "Unknown traffic"}
    if label_lower in label_map:
        return label_map[label_lower]
    if "safeframe" in label_lower or "googlesyndication" in label_lower:
        return "Google ad frame referral traffic"
    if "googleusercontent" in label_lower:
        return "Google hosted referral traffic"
    if " / " in label:
        source, medium = label.split(" / ", 1)
        source_clean = source.strip().lower()
        medium_clean = medium.strip().lower()
        if source_clean in ["<other>", "other"]:
            return f"Other {medium_clean} traffic" if medium_clean not in ["<other>", "other"] else "Other channels"
        if source_clean in ["(not set)", "(data deleted)", "data deleted"]:
            return f"Unknown {medium_clean} traffic" if medium_clean not in ["(not set)", "(data deleted)"] else "Unknown traffic"
        if len(source_clean) > 45 or source_clean.count(".") >= 3:
            return f"Other {medium_clean} traffic" if medium_clean not in ["(not set)", "(none)"] else "Other traffic"
        return f"{source_clean.replace('_', ' ').replace('-', ' ').title()} {medium_clean.replace('_', ' ')} traffic"
    return label.replace("_", " ").replace("-", " ").title()

# Map funnel transitions to owner and action fields
def build_action_metadata(row):
    transition = (row.get("step_from"), row.get("step_to"))
    action_map = {
        ("view_item", "add_to_cart"): ("Product discovery", "Product page optimisation", "Product / UX"),
        ("add_to_cart", "begin_checkout"): ("Cart", "Cart friction review", "UX / Ecommerce"),
        ("begin_checkout", "add_shipping_info"): ("Checkout", "Checkout form review", "UX / Engineering"),
        ("add_shipping_info", "add_payment_info"): ("Shipping to payment", "Cost and payment-step review", "UX / Engineering"),
        ("add_payment_info", "purchase"): ("Payment completion", "Payment completion review", "Engineering / Payments")}
    funnel_area, action_category, suggested_owner = action_map.get(transition, ("Funnel transition", "Funnel review", "Product / Analytics"))
    return {"funnel_area": funnel_area, "action_category": action_category, "suggested_owner": suggested_owner}

# Build suggested action text for each issue
def build_action_hint(row):
    label = clean_display_label(row)
    step_from, step_to = row.get("step_from"), row.get("step_to")
    if step_from == "view_item" and step_to == "add_to_cart":
        return f"Review product-page clarity, pricing visibility, and add-to-cart prompts for {label}."
    if step_from == "add_to_cart" and step_to == "begin_checkout":
        return f"Review cart-to-checkout friction, delivery messaging, and checkout call-to-action clarity for {label}."
    if step_from == "begin_checkout" and step_to == "add_shipping_info":
        return f"Review checkout form clarity and shipping information expectations for {label}."
    if step_from == "add_shipping_info" and step_to == "add_payment_info":
        return f"Review shipping-to-payment flow, cost transparency, and payment-step messaging for {label}."
    if step_from == "add_payment_info" and step_to == "purchase":
        return f"Review payment completion, error handling, and final confirmation friction for {label}."
    return f"Review the {step_display(step_from)} to {step_display(step_to)} transition for {label}."

# Build the longer evidence sentence
def build_evidence_text(row):
    gap = row.get("conversion_gap", np.nan)
    lost = row.get("estimated_lost_purchases", np.nan)
    current = row.get("current_step_conversion", np.nan)
    baseline = row.get("baseline_step_conversion", np.nan)
    earlier = row.get("earlier_step_sessions", np.nan)
    return (f"Current conversion was {pct_text(current)} versus a weekly baseline of {pct_text(baseline)}, "
            f"a gap of {pct_text(gap)} across {num_text(earlier, 0)} earlier-step sessions. "
            f"Estimated lost purchases: {num_text(lost, 1)}.")

# Build the shorter dashboard evidence sentence
def build_short_evidence_text(row):
    lost = row.get("estimated_lost_purchases", np.nan)
    gap = row.get("conversion_gap", np.nan)
    return f"{num_text(lost, 1)} estimated lost purchases; {pct_text(gap)} conversion gap."

# Label whether an issue is new or recurring
def build_issue_status(row):
    if pd.isna(row.get("previous_rank", np.nan)):
        return "New issue"
    if row["rank"] < row["previous_rank"]:
        return "Moved up in priority"
    if row["rank"] > row["previous_rank"]:
        return "Moved down in priority"
    return "Recurring issue"

# Build readable weekly trend text
def build_trend_text(row):
    if pd.isna(row.get("previous_rank", np.nan)):
        return "New in the latest weekly Top Issues list." if row.get("is_latest_week", False) else "No previous-week match found."
    rank_text = f"rank {int(row['previous_rank'])} to rank {int(row['rank'])}"
    if row["rank"] < row["previous_rank"]:
        return f"Moved up from {rank_text}."
    if row["rank"] > row["previous_rank"]:
        return f"Moved down from {rank_text}."
    return f"Recurring at rank {int(row['rank'])}."


In [4]:
#------------------------------------------------------------------------------
# 2.3 Load final reporting inputs
#------------------------------------------------------------------------------
print_banner("2.3 Load final reporting inputs")

window_start, window_end = START_DATE, END_DATE

# Load final CSV inputs
funnel_weekly_rates = clean_date_col(read_csv_required(FUNNEL_WEEKLY_PATH))
segment_step_metrics = clean_date_col(read_csv_required(SEGMENT_STEP_PATH))
top_issues_weekly = clean_date_col(read_csv_required(TOP_ISSUES_PATH))
model_comparison = read_csv_required(MODEL_COMPARISON_PATH)
ranking_cutoff_metrics = read_csv_required(RANKING_CUTOFF_PATH)
grouped_model_importance = read_csv_required(GROUPED_IMPORTANCE_PATH)
combined_candidates = read_csv_required(COMBINED_CANDIDATES_PATH)

# Clean reporting labels for text fields
for df in [top_issues_weekly, segment_step_metrics]:
    if "segment_value" in df.columns:
        if "segment_value_display" not in df.columns:
            df["segment_value_display"] = df["segment_value"]
        df["segment_value_display"] = df.apply(clean_display_label, axis=1)

# Check required columns
require_cols(funnel_weekly_rates, ["week_start_date"], "funnel_weekly_rates")
require_cols(top_issues_weekly, ["week_start_date", "issue_label", "issue_rank_within_week", "step_from", "step_to"], "top_issues_weekly")
require_cols(model_comparison, ["data_split", "model", "pr_auc", "brier_score"], "model_comparison")
require_cols(ranking_cutoff_metrics, ["data_split", "model", "cutoff_label", "precision_at_k", "recall_at_k", "lift_at_k"], "ranking_cutoff_metrics")
require_cols(grouped_model_importance, ["feature_group", "importance"], "grouped_model_importance")
require_cols(combined_candidates, ["intervention_group", "purchase_likelihood_score", "within_group_priority_score", "within_group_priority_rank", "within_group_priority_band", "channel_segment_clean"], "combined_candidates")

# Keep Notebook 03 score field names
combined_candidates = combined_candidates.copy()
score_col = "purchase_likelihood_score"
priority_col = "within_group_priority_score"
rank_col = "within_group_priority_rank"
band_col = "within_group_priority_band"
channel_col = "channel_segment_clean"

# Identify the latest reporting week
latest_week = str(pd.to_datetime(top_issues_weekly["week_start_date"], errors="coerce").max().date())
print("Project date window:", window_start, "to", window_end)
print("Latest reporting week:", latest_week)

# Summarise loaded input sizes
input_check_df = pd.DataFrame([
    {"input_name":"funnel_weekly_rates", "rows":len(funnel_weekly_rates), "columns":funnel_weekly_rates.shape[1]},
    {"input_name":"segment_step_metrics_weekly", "rows":len(segment_step_metrics), "columns":segment_step_metrics.shape[1]},
    {"input_name":"top_issues_weekly", "rows":len(top_issues_weekly), "columns":top_issues_weekly.shape[1]},
    {"input_name":"model_comparison_metrics", "rows":len(model_comparison), "columns":model_comparison.shape[1]},
    {"input_name":"ranking_cutoff_metrics", "rows":len(ranking_cutoff_metrics), "columns":ranking_cutoff_metrics.shape[1]},
    {"input_name":"grouped_model_importance", "rows":len(grouped_model_importance), "columns":grouped_model_importance.shape[1]},
    {"input_name":"combined_priority_candidates", "rows":len(combined_candidates), "columns":combined_candidates.shape[1]}])
display(input_check_df)



------------------------------------------------------------------------------
2.3 Load final reporting inputs
------------------------------------------------------------------------------
Project date window: 20201101 to 20210228
Latest reporting week: 2021-01-25


,input_name,rows,columns
0,funnel_weekly_rates,14,25
1,segment_step_metrics_weekly,11755,22
2,top_issues_weekly,41,18
3,model_comparison_metrics,6,15
4,ranking_cutoff_metrics,30,15
5,grouped_model_importance,6,3
6,combined_priority_candidates,200,51


# 3.0 Build final reporting summaries

The final reporting summaries collect the evidence used in the deterministic memo and Power BI decision page. These tables stay compact and focus on the information needed to explain the latest funnel issues, the quality of the model ranking, the main feature signals, and the role of each intervention review group.

**Process**:
1. Select the latest reporting week and keep the Top 3 issues for that week.
2. Identify the held-out test model used for reporting, preferring calibrated scores when model performance is tied.
3. Pull the final model ranking cut-offs to show whether the model produces useful review shortlists.
4. Summarise grouped feature importance to explain which signal types drive the purchase likelihood model.
5. Summarise intervention review groups using purchase likelihood and within group priority fields from Notebook 03.
6. Display small evidence tables that can be reused directly in the memo and Power BI decision summary.


In [5]:
#------------------------------------------------------------------------------
# 3.1 Latest Top Issues and model evidence
#------------------------------------------------------------------------------
print_banner("3.1 Latest Top Issues and model evidence")

# Keep the latest week's top ranked issues
latest_week_top3 = (top_issues_weekly[top_issues_weekly["week_start_date"].astype(str).eq(latest_week)].sort_values("issue_rank_within_week").head(3).copy())
latest_week_top3["evidence_text"] = latest_week_top3.apply(build_evidence_text, axis=1)
latest_week_top3["suggested_action"] = latest_week_top3.apply(build_action_hint, axis=1)

# Select the best held-out model
test_models = model_comparison[model_comparison["data_split"].eq("test")].copy()
test_models["is_calibrated"] = test_models["model"].astype(str).str.contains("calibrated", case=False, na=False).astype(int)
best_test_model = (test_models.sort_values(["pr_auc", "is_calibrated", "brier_score"], ascending=[False, False, True]).iloc[0].copy())

# Pull ranking metrics for the selected model
final_model_name = str(best_test_model["model"])
test_ranking = ranking_cutoff_metrics[(ranking_cutoff_metrics["data_split"].eq("test")) &
                                      (ranking_cutoff_metrics["model"].eq(final_model_name))].copy()

# Fall back to calibrated test ranking if needed
if len(test_ranking) == 0:
    calibrated_names = ranking_cutoff_metrics[ranking_cutoff_metrics["data_split"].eq("test")]["model"].astype(str)
    final_model_name = calibrated_names[calibrated_names.str.contains("calibrated", case=False, na=False)].iloc[-1]
    test_ranking = ranking_cutoff_metrics[(ranking_cutoff_metrics["data_split"].eq("test")) &
                                          (ranking_cutoff_metrics["model"].eq(final_model_name))].copy()

# Sort ranking cut-offs in report order
cutoff_order = {"top_1pct":1, "top_2pct":2, "top_5pct":3, "top_100":4, "top_250":5}
test_ranking["cutoff_order"] = test_ranking["cutoff_label"].map(cutoff_order).fillna(99)
test_ranking = test_ranking.sort_values("cutoff_order").drop(columns=["cutoff_order"])

# Calculate importance shares if needed
importance_summary = grouped_model_importance.copy()
if "importance_share" not in importance_summary.columns and importance_summary["importance"].sum() > 0:
    importance_summary["importance_share"] = importance_summary["importance"] / importance_summary["importance"].sum()
importance_summary = importance_summary.sort_values("importance", ascending=False)

# Collect model metrics for the summary table
top1_row = test_ranking[test_ranking["cutoff_label"].eq("top_1pct")].head(1)
top100_row = test_ranking[test_ranking["cutoff_label"].eq("top_100")].head(1)
model_evidence = {
    "model": final_model_name,
    "test_pr_auc": float(best_test_model.get("pr_auc", np.nan)),
    "test_roc_auc": float(best_test_model.get("roc_auc", np.nan)) if "roc_auc" in best_test_model.index else None,
    "test_brier_score": float(best_test_model.get("brier_score", np.nan)),
    "top_1pct_precision": float(top1_row["precision_at_k"].iloc[0]) if len(top1_row) else None,
    "top_1pct_recall": float(top1_row["recall_at_k"].iloc[0]) if len(top1_row) else None,
    "top_100_precision": float(top100_row["precision_at_k"].iloc[0]) if len(top100_row) else None,
    "top_100_recall": float(top100_row["recall_at_k"].iloc[0]) if len(top100_row) else None}

# Keep display columns that exist
issue_display_cols = ["issue_rank_within_week", "issue_label", "segment_type", "segment_value_display", "step_from", "step_to", "estimated_lost_purchases", "conversion_gap"]
issue_display_cols = [c for c in issue_display_cols if c in latest_week_top3.columns]
ranking_display_cols = ["cutoff_label", "k_n", "precision_at_k", "recall_at_k", "lift_at_k", "positives_captured_n", "n_positive"]
ranking_display_cols = [c for c in ranking_display_cols if c in test_ranking.columns]

print("Latest Top 3 issues")
display(latest_week_top3[issue_display_cols])
print("\n")
print("Best held-out model")
display(pd.DataFrame([best_test_model.drop(labels=["is_calibrated"], errors="ignore")]))
print("\n")
print("Final model ranking cut-offs")
display(test_ranking[ranking_display_cols])
print("\n")
print("Grouped model importance")
display(importance_summary.head(8))



------------------------------------------------------------------------------
3.1 Latest Top Issues and model evidence
------------------------------------------------------------------------------
Latest Top 3 issues


,issue_rank_within_week,issue_label,segment_type,segment_value_display,step_from,step_to,estimated_lost_purchases,conversion_gap
38,1,Google organic traffic: product view to cart drop-off,channel_segment,Google Organic Traffic,view_item,add_to_cart,58.1238,0.0591
39,2,Google organic traffic: shipping to payment drop-off,channel_segment,Google Organic Traffic,add_shipping_info,add_payment_info,19.6257,0.1372
40,3,Google organic traffic: cart to checkout drop-off,channel_segment,Google Organic Traffic,add_to_cart,begin_checkout,9.1728,0.0428




Best held-out model


,split,data_split,model,n_obs,n_positive,prevalence,pr_auc,roc_auc,brier_score,top_frac,k_n,precision_at_k,recall_at_k,lift_at_k,model_display_name
5,test,test,lightgbm_calibrated,26342,310,0.0118,0.8039,0.9984,0.0031,0.0500,1318,0.2352,1.0000,19.9863,LightGBM calibrated likelihood




Final model ranking cut-offs


,cutoff_label,k_n,precision_at_k,recall_at_k,lift_at_k,positives_captured_n,n_positive
25,top_1pct,264,0.8106,0.6903,68.8806,214,310
26,top_2pct,527,0.5882,1.0000,49.9848,310,310
27,top_5pct,1318,0.2352,1.0000,19.9863,310,310
28,top_100,100,0.8500,0.2742,72.2281,85,310
29,top_250,250,0.8160,0.6581,69.3389,204,310




Grouped model importance


,feature_group,importance,importance_share
0,session behaviour,138,0.4600
1,item intensity,61,0.2033
2,funnel progression,45,0.1500
3,device,22,0.0733
4,geography,18,0.0600
5,traffic source / medium,16,0.0533


In [6]:
#------------------------------------------------------------------------------
# 3.2 Intervention review group summaries
#------------------------------------------------------------------------------
print_banner("3.2 Intervention review group summaries")

# Summarise each intervention group
candidate_key = "user_session_key" if "user_session_key" in combined_candidates.columns else "intervention_group"
intervention_summary = (combined_candidates.groupby("intervention_group", as_index=False)
                        .agg(n_candidates=(candidate_key, "nunique"),
                             avg_purchase_likelihood_score=(score_col, "mean"),
                             median_purchase_likelihood_score=(score_col, "median"),
                             max_purchase_likelihood_score=(score_col, "max"),
                             avg_priority_within_group=(priority_col, "mean")))

# Add interpretation and display order
interpretation_map = {
    "Browse drop-off review": "Diagnostic review list for early product-view drop-off; not a high-likelihood recovery group.",
    "Cart drop-off review": "Diagnostic review list for cart-stage friction; not a high-likelihood recovery group.",
    "Late checkout recovery": "Model-led recovery list for sessions with much higher purchase likelihood."}
intervention_sort_map = {"Browse drop-off review":1, "Cart drop-off review":2, "Late checkout recovery":3}
intervention_summary["interpretation"] = intervention_summary["intervention_group"].map(interpretation_map).fillna("Intervention review group.")
intervention_summary["intervention_sort_order"] = intervention_summary["intervention_group"].map(intervention_sort_map).fillna(99).astype(int)
intervention_summary = intervention_summary.sort_values("intervention_sort_order")

# Summarise channel mix within intervention groups
intervention_channel_mix = (combined_candidates.groupby(["intervention_group", channel_col], as_index=False).agg(n_candidates=(candidate_key, "nunique"),
                              avg_purchase_likelihood_score=(score_col, "mean")).sort_values(["intervention_group", "n_candidates"], ascending=[True, False]))

# Keep top priority rows within each group
top_intervention_rows = combined_candidates.copy()
top_intervention_rows["_sort_rank"] = top_intervention_rows[rank_col] if rank_col in top_intervention_rows.columns else top_intervention_rows[priority_col].rank(ascending=False)
top_intervention_rows = (top_intervention_rows.sort_values(["intervention_group", "_sort_rank"])
                         .groupby("intervention_group", as_index=False)
                         .head(5)
                         .drop(columns=["_sort_rank"], errors="ignore"))

# Keep candidate columns that exist
candidate_display_cols = ["intervention_group", score_col, priority_col, rank_col, band_col, "priority_score_scope", "device_category", "geo_country", channel_col, "last_observed_funnel_stage", "dropoff_after_stage"]
candidate_display_cols = [c for c in candidate_display_cols if c in top_intervention_rows.columns]

display(intervention_summary)
print("Largest intervention groups by channel")
display(intervention_channel_mix.head(20))
print("Top rows within each intervention group")
display(top_intervention_rows[candidate_display_cols].head(15))



------------------------------------------------------------------------------
3.2 Intervention review group summaries
------------------------------------------------------------------------------


,intervention_group,n_candidates,avg_purchase_likelihood_score,median_purchase_likelihood_score,max_purchase_likelihood_score,avg_priority_within_group,interpretation,intervention_sort_order
0,Browse drop-off review,50,0.0001,0.0000,0.0025,0.5000,Diagnostic review list for early product-view drop-off; not a high-likelihood recovery group.,1
1,Cart drop-off review,50,0.0000,0.0000,0.0000,0.5000,Diagnostic review list for cart-stage friction; not a high-likelihood recovery group.,2
2,Late checkout recovery,100,0.5992,0.7155,0.7349,0.5000,Model-led recovery list for sessions with much higher purchase likelihood.,3


Largest intervention groups by channel


,intervention_group,channel_segment_clean,n_candidates,avg_purchase_likelihood_score
4,Browse drop-off review,Google organic traffic,29,0.0001
3,Browse drop-off review,Google Merch Store referral traffic,5,0.0000
5,Browse drop-off review,Other referral traffic,5,0.0000
2,Browse drop-off review,Google Analytics referral traffic,4,0.0001
1,Browse drop-off review,Direct traffic,2,0.0000
0,Browse drop-off review,Coursera.Org referral traffic,1,0.0000
6,Browse drop-off review,Partners affiliate traffic,1,0.0000
7,Browse drop-off review,Perks at Work referral traffic,1,0.0000
8,Browse drop-off review,Unknown traffic,1,0.0000
9,Browse drop-off review,YouTube Creator Academy referral traffic,1,0.0000


Top rows within each intervention group


,intervention_group,purchase_likelihood_score,within_group_priority_score,within_group_priority_rank,within_group_priority_band,priority_score_scope,device_category,geo_country,channel_segment_clean,last_observed_funnel_stage,dropoff_after_stage
0,Browse drop-off review,0.0025,1.0000,1,very_high,within_intervention_group_percentile,desktop,Colombia,Google organic traffic,view_item,after_view_item
1,Browse drop-off review,0.0001,0.9796,2,very_high,within_intervention_group_percentile,mobile,United States,Google Analytics referral traffic,view_item,after_view_item
2,Browse drop-off review,0.0001,0.9592,3,very_high,within_intervention_group_percentile,mobile,United States,Google Analytics referral traffic,view_item,after_view_item
3,Browse drop-off review,0.0001,0.9388,4,very_high,within_intervention_group_percentile,mobile,South Korea,Google Analytics referral traffic,view_item,after_view_item
4,Browse drop-off review,0.0001,0.9184,5,very_high,within_intervention_group_percentile,desktop,United States,Google Analytics referral traffic,view_item,after_view_item
50,Cart drop-off review,0.0000,1.0000,1,very_high,within_intervention_group_percentile,desktop,India,Google organic traffic,add_to_cart,after_add_to_cart
51,Cart drop-off review,0.0000,0.9796,2,very_high,within_intervention_group_percentile,desktop,United States,Google Merch Store referral traffic,add_to_cart,after_add_to_cart
52,Cart drop-off review,0.0000,0.9592,3,very_high,within_intervention_group_percentile,desktop,United States,Google Merch Store referral traffic,add_to_cart,after_add_to_cart
53,Cart drop-off review,0.0000,0.9388,4,very_high,within_intervention_group_percentile,desktop,India,Google organic traffic,add_to_cart,after_add_to_cart
54,Cart drop-off review,0.0000,0.9184,5,very_high,within_intervention_group_percentile,desktop,United States,Google organic traffic,add_to_cart,after_add_to_cart


# 4.0 Create weekly decision summary for Power BI

The weekly decision summary is the main Power BI table produced by Notebook 04. It combines Top Issues, model evidence, and intervention review group explanations into one refreshable table, so the dashboard can use dynamic text, trend labels, action categories, and visual sort fields instead of manually written report notes.

**Process**:
1. Start from `top_issues_weekly`, keeping one row per ranked issue and week.
2. Build issue keys and compare each issue with the previous week to create status, rank change, and trend text fields.
3. Add dashboard headline, evidence, action, funnel area, suggested owner, and priority fields.
4. Append model evidence rows for test PR-AUC, Brier score, top list precision, and the main feature group.
5. Append intervention group rows that explain browse review, cart review, and late checkout recovery in the same table.
6. Save only `weekly_decision_summary.csv` to the Power BI latest folder.


In [7]:
#------------------------------------------------------------------------------
# 4.1 Build and save weekly decision summary
#------------------------------------------------------------------------------
print_banner("4.1 Build and save weekly decision summary")

# Build Top Issue rows
decision_df = top_issues_weekly.copy().sort_values(["week_start_date", "issue_rank_within_week"])
decision_df["week_dt"] = pd.to_datetime(decision_df["week_start_date"], errors="coerce")
decision_df["summary_type"] = "top_issue"
decision_df["dashboard_section"] = "Top Issues"
decision_df["rank"] = pd.to_numeric(decision_df["issue_rank_within_week"], errors="coerce").astype("Int64")
decision_df["issue_key"] = decision_df.apply(build_issue_key, axis=1)

# Convert issue metrics to numeric values
for col in ["estimated_lost_purchases", "conversion_gap", "current_step_conversion", "baseline_step_conversion", "earlier_step_sessions"]:
    if col in decision_df.columns:
        decision_df[col] = pd.to_numeric(decision_df[col], errors="coerce")

# Create previous-week comparison fields
prev_cols = ["issue_key", "week_dt", "rank", "estimated_lost_purchases", "conversion_gap"]
prev_df = decision_df[[c for c in prev_cols if c in decision_df.columns]].copy()
prev_df["week_dt"] = prev_df["week_dt"] + pd.Timedelta(days=7)
prev_df = prev_df.rename(columns={"rank":"previous_rank", "estimated_lost_purchases":"previous_estimated_lost_purchases", "conversion_gap":"previous_conversion_gap"})
decision_df = decision_df.merge(prev_df, on=["issue_key", "week_dt"], how="left")

# Add trend and status fields
decision_df["rank_change"] = decision_df["previous_rank"] - decision_df["rank"]
decision_df["lost_purchases_change"] = decision_df.get("estimated_lost_purchases", np.nan) - decision_df.get("previous_estimated_lost_purchases", np.nan)
decision_df["conversion_gap_change"] = decision_df.get("conversion_gap", np.nan) - decision_df.get("previous_conversion_gap", np.nan)
decision_df["is_latest_week"] = decision_df["week_start_date"].astype(str).eq(latest_week)
decision_df["issue_status"] = decision_df.apply(build_issue_status, axis=1)
decision_df["trend_text"] = decision_df.apply(build_trend_text, axis=1)

# Add dashboard text and ownership fields
action_meta_df = decision_df.apply(lambda row: pd.Series(build_action_metadata(row)), axis=1)
decision_df = pd.concat([decision_df, action_meta_df], axis=1)
decision_df["headline_text"] = decision_df["issue_label"]
decision_df["evidence_metric"] = np.where(decision_df.get("estimated_lost_purchases", pd.Series(index=decision_df.index)).notna(), "estimated_lost_purchases", "conversion_gap")
decision_df["evidence_value"] = np.where(decision_df["evidence_metric"].eq("estimated_lost_purchases"), decision_df.get("estimated_lost_purchases", np.nan), decision_df.get("conversion_gap", np.nan))
decision_df["evidence_text"] = decision_df.apply(build_evidence_text, axis=1)
decision_df["short_evidence_text"] = decision_df.apply(build_short_evidence_text, axis=1)
decision_df["suggested_action"] = decision_df.apply(build_action_hint, axis=1)
decision_df["dashboard_note"] = decision_df["trend_text"]
decision_df["priority"] = decision_df["rank"].map({1:"high", 2:"medium", 3:"review"}).fillna("review")
decision_df["priority_numeric"] = decision_df["rank"].astype(float)
decision_df["visual_sort_order"] = decision_df["rank"].astype(float)

# Add model evidence rows
main_feature = importance_summary.iloc[0] if len(importance_summary) else pd.Series(dtype=object)
model_rows = [
    {"summary_type":"model_evidence", "dashboard_section":"Model Evidence", "visual_sort_order":101, "rank":1, "headline_text":"Held-out test PR-AUC", "evidence_metric":"test_pr_auc", "evidence_value":model_evidence["test_pr_auc"], "evidence_text":f"The final model achieved {model_evidence['test_pr_auc']:.4f} PR-AUC on the held-out test set.", "short_evidence_text":f"Test PR-AUC: {model_evidence['test_pr_auc']:.4f}", "suggested_action":"Use model scores as supporting evidence for intervention prioritisation.", "dashboard_note":"Model evidence is based on the held-out test split.", "priority":"evidence", "priority_numeric":4, "funnel_area":"Model", "action_category":"Model evidence", "suggested_owner":"Analytics / Product"},
    {"summary_type":"model_evidence", "dashboard_section":"Model Evidence", "visual_sort_order":102, "rank":2, "headline_text":"Brier score", "evidence_metric":"test_brier_score", "evidence_value":model_evidence["test_brier_score"], "evidence_text":f"The calibrated model produced a Brier score of {model_evidence['test_brier_score']:.4f}.", "short_evidence_text":f"Brier score: {model_evidence['test_brier_score']:.4f}", "suggested_action":"Use calibrated scores as directional likelihood evidence, not causal proof.", "dashboard_note":"Lower Brier score indicates better probability calibration.", "priority":"evidence", "priority_numeric":4, "funnel_area":"Model", "action_category":"Calibration evidence", "suggested_owner":"Analytics"},
    {"summary_type":"model_evidence", "dashboard_section":"Model Evidence", "visual_sort_order":103, "rank":3, "headline_text":"Top 1% precision", "evidence_metric":"top_1pct_precision", "evidence_value":model_evidence["top_1pct_precision"], "evidence_text":f"The top 1% review list reached {pct_text(model_evidence['top_1pct_precision'])} precision and {pct_text(model_evidence['top_1pct_recall'])} recall.", "short_evidence_text":f"Top 1% precision: {pct_text(model_evidence['top_1pct_precision'])}", "suggested_action":"Use top-ranked sessions to inspect high-likelihood recovery patterns.", "dashboard_note":"Ranking metrics show whether the model creates useful shortlists.", "priority":"evidence", "priority_numeric":4, "funnel_area":"Model", "action_category":"Ranking evidence", "suggested_owner":"Analytics / Growth"},
    {"summary_type":"model_evidence", "dashboard_section":"Model Evidence", "visual_sort_order":104, "rank":4, "headline_text":"Top 100 precision", "evidence_metric":"top_100_precision", "evidence_value":model_evidence["top_100_precision"], "evidence_text":f"The top 100 review list reached {pct_text(model_evidence['top_100_precision'])} precision and {pct_text(model_evidence['top_100_recall'])} recall.", "short_evidence_text":f"Top 100 precision: {pct_text(model_evidence['top_100_precision'])}", "suggested_action":"Use the Top 100 shortlist as the clearest model-led recovery review list.", "dashboard_note":"Top 100 precision is easy to interpret for stakeholder-facing reporting.", "priority":"evidence", "priority_numeric":4, "funnel_area":"Model", "action_category":"Ranking evidence", "suggested_owner":"Analytics / Growth"},
    {"summary_type":"model_evidence", "dashboard_section":"Model Evidence", "visual_sort_order":105, "rank":5, "headline_text":"Top feature group", "evidence_metric":"main_feature_group_importance_share", "evidence_value":main_feature.get("importance_share", np.nan), "evidence_text":f"The largest feature group was {main_feature.get('feature_group', 'not available')}, supporting the behavioural decision-tool framing.", "short_evidence_text":f"Main signal: {main_feature.get('feature_group', 'not available')}", "suggested_action":"Interpret model scores alongside funnel diagnostics and feature-group evidence.", "dashboard_note":"Feature importance is grouped to keep model interpretation readable.", "priority":"evidence", "priority_numeric":4, "funnel_area":"Model", "action_category":"Interpretability evidence", "suggested_owner":"Analytics"}]

# Add intervention group rows
intervention_rows = []
for _, row in intervention_summary.sort_values("intervention_sort_order").iterrows():
    group = row["intervention_group"]
    is_late = group == "Late checkout recovery"
    action = "Prioritise these sessions for payment and checkout recovery review." if is_late else "Use this group to inspect earlier funnel friction by segment, device, and channel."
    note = row["interpretation"]
    evidence = f"{int(row['n_candidates']):,} sessions identified; median purchase likelihood was {pct_text(row['median_purchase_likelihood_score'])}."
    intervention_rows.append({"summary_type":"intervention_group", "dashboard_section":"Intervention Groups", "visual_sort_order":200 + int(row["intervention_sort_order"]), "rank":int(row["intervention_sort_order"]), "headline_text":group, "issue_label":group, "evidence_metric":"n_candidates", "evidence_value":row["n_candidates"], "evidence_text":evidence, "short_evidence_text":evidence, "suggested_action":action, "dashboard_note":note, "priority":"review" if not is_late else "high", "priority_numeric":2 if is_late else 3, "funnel_area":"Intervention review", "action_category":"Recovery review" if is_late else "Diagnostic review", "suggested_owner":"Growth / Ecommerce" if is_late else "Product / UX"})

final_cols = ["week_start_date", "summary_type", "dashboard_section", "visual_sort_order", "rank", "issue_key", "issue_status", "previous_rank", "rank_change", "lost_purchases_change", "conversion_gap_change", "trend_text", "issue_label", "headline_text", "segment_type", "segment_value", "segment_value_display", "step_from", "step_to", "funnel_area", "action_category", "suggested_owner", "evidence_metric", "evidence_value", "evidence_text", "short_evidence_text", "suggested_action", "dashboard_note", "priority", "priority_numeric", "is_latest_week"]

# Align all row types to the same schema
def align_decision_cols(df):
    out = df.copy()
    out["week_start_date"] = out.get("week_start_date", latest_week)
    out["is_latest_week"] = out.get("is_latest_week", True)
    for col in final_cols:
        if col not in out.columns:
            out[col] = np.nan
    return out[final_cols]

# Align issue, model and intervention rows
issue_rows = align_decision_cols(decision_df)
model_decision_rows = align_decision_cols(pd.DataFrame(model_rows))
intervention_decision_rows = align_decision_cols(pd.DataFrame(intervention_rows))

# Combine all decision summary rows
decision_frames = [issue_rows, model_decision_rows, intervention_decision_rows]
decision_frames = [df for df in decision_frames if len(df) > 0]

if len(decision_frames) == 0:
    weekly_decision_summary = pd.DataFrame(columns=final_cols)
else:
    weekly_decision_summary = pd.concat(decision_frames, ignore_index=True)
    weekly_decision_summary = weekly_decision_summary[final_cols]

# Sort and save the final summary
weekly_decision_summary = weekly_decision_summary.sort_values(
    ["week_start_date", "visual_sort_order", "summary_type"], na_position="last")

save_csv(weekly_decision_summary, WEEKLY_DECISION_SUMMARY_PATH)

# Prepare latest-week checks
latest_decision_summary = weekly_decision_summary[weekly_decision_summary["week_start_date"].astype(str).eq(latest_week)].copy()
latest_decision_counts = weekly_decision_summary.groupby("summary_type", as_index=False).size().rename(columns={"size":"n_rows"})

display(latest_decision_summary[[c for c in ["summary_type", "dashboard_section", "rank", "headline_text", "issue_status", "trend_text", "short_evidence_text", "suggested_action"] if c in latest_decision_summary.columns]].head(20))
display(latest_decision_counts)
print("Saved weekly decision summary ->", WEEKLY_DECISION_SUMMARY_PATH)



------------------------------------------------------------------------------
4.1 Build and save weekly decision summary
------------------------------------------------------------------------------


/tmp/ipykernel_1289/3606927901.py:91: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  weekly_decision_summary = pd.concat(decision_frames, ignore_index=True)


,summary_type,dashboard_section,rank,headline_text,issue_status,trend_text,short_evidence_text,suggested_action
38,top_issue,Top Issues,1,Google organic traffic: product view to cart drop-off,Moved up in priority,Moved up from rank 2 to rank 1.,58.1 estimated lost purchases; 5.9% conversion gap.,"Review product-page clarity, pricing visibility, and add-to-cart prompts for Google Organic Traffic."
39,top_issue,Top Issues,2,Google organic traffic: shipping to payment drop-off,Moved down in priority,Moved down from rank 1 to rank 2.,19.6 estimated lost purchases; 13.7% conversion gap.,"Review shipping-to-payment flow, cost transparency, and payment-step messaging for Google Organic Traffic."
40,top_issue,Top Issues,3,Google organic traffic: cart to checkout drop-off,New issue,New in the latest weekly Top Issues list.,9.2 estimated lost purchases; 4.3% conversion gap.,"Review cart-to-checkout friction, delivery messaging, and checkout call-to-action clarity for Google Organic Traffic."
41,model_evidence,Model Evidence,1,Held-out test PR-AUC,NaN,NaN,Test PR-AUC: 0.8039,Use model scores as supporting evidence for intervention prioritisation.
42,model_evidence,Model Evidence,2,Brier score,NaN,NaN,Brier score: 0.0031,"Use calibrated scores as directional likelihood evidence, not causal proof."
43,model_evidence,Model Evidence,3,Top 1% precision,NaN,NaN,Top 1% precision: 81.1%,Use top-ranked sessions to inspect high-likelihood recovery patterns.
44,model_evidence,Model Evidence,4,Top 100 precision,NaN,NaN,Top 100 precision: 85.0%,Use the Top 100 shortlist as the clearest model-led recovery review list.
45,model_evidence,Model Evidence,5,Top feature group,NaN,NaN,Main signal: session behaviour,Interpret model scores alongside funnel diagnostics and feature-group evidence.
46,intervention_group,Intervention Groups,1,Browse drop-off review,NaN,NaN,50 sessions identified; median purchase likelihood was 0.0%.,"Use this group to inspect earlier funnel friction by segment, device, and channel."
47,intervention_group,Intervention Groups,2,Cart drop-off review,NaN,NaN,50 sessions identified; median purchase likelihood was 0.0%.,"Use this group to inspect earlier funnel friction by segment, device, and channel."


,summary_type,n_rows
0,intervention_group,3
1,model_evidence,5
2,top_issue,41


Saved weekly decision summary -> /content/drive/MyDrive/Capstone_Project/powerbi_extracts/latest/weekly_decision_summary.csv


# 5.0 Save deterministic weekly memo

The memo uses the same evidence rows as the Power BI decision summary, keeping the final reporting layer consistent. It remains deterministic, so the output is stable, inspectable, and suitable for submission

**Process**:
1. Write a concise weekly memo that highlights the latest issues, their trend status, model evidence, and intervention review interpretation.
2. Use careful wording for intervention groups so browse/cart review lists are not overstated as high likelihood recovery groups.
3. Save the memo with a date stamped filename for reproducibility.


In [8]:
#------------------------------------------------------------------------------
# 5.1 Build deterministic weekly memo
#------------------------------------------------------------------------------
print_banner("5.1 Build deterministic weekly memo")

# Keep latest top issue rows
memo_top_issues = latest_decision_summary[latest_decision_summary["summary_type"].eq("top_issue")].sort_values("rank")

# Start memo heading
memo_lines = [f"# Weekly GA4 decision memo — week starting {latest_week}", "", "## 1. Latest Top Issues", ""]

# Add latest Top Issues
for _, row in memo_top_issues.head(3).iterrows():
    memo_lines.extend([
        f"{int(row['rank'])}. **{row['headline_text']}**",
        f"   - Evidence: {row['evidence_text']}",
        f"   - Trend: {row['trend_text']}",
        f"   - Suggested action: {row['suggested_action']}",
        ""])

# Add model evidence
memo_lines.extend([
    "## 2. Model evidence", "",
    f"- The final session-level purchase propensity and prioritisation model is `{final_model_name}`.",
    f"- Held-out test PR-AUC is {model_evidence['test_pr_auc']:.4f} and Brier score is {model_evidence['test_brier_score']:.4f}.",
    f"- The top 1% review list reached {pct_text(model_evidence['top_1pct_precision'])} precision and {pct_text(model_evidence['top_1pct_recall'])} recall.",
    f"- The top 100 review list reached {pct_text(model_evidence['top_100_precision'])} precision and {pct_text(model_evidence['top_100_recall'])} recall.",
    "", "## 3. Intervention review groups", ""])

# Add intervention group notes
for _, row in intervention_summary.iterrows():
    memo_lines.append(f"- **{row['intervention_group']}:** {int(row['n_candidates']):,} sessions; median purchase likelihood {pct_text(row['median_purchase_likelihood_score'])}. {row['interpretation']}")

# Add reporting note
memo_lines.extend([
    "", "## 4. Reporting note", "",
    "Browse and cart review groups support funnel diagnosis, while late checkout recovery is the strongest model-led recovery group. The Power BI decision summary keeps raw segment fields for filtering, readable labels for reporting, and dynamic trend fields for weekly interpretation."])

# Save memo text
weekly_memo_text = "\n".join(memo_lines)
WEEKLY_MEMO_PATH.write_text(weekly_memo_text, encoding="utf-8")

display(Markdown(weekly_memo_text))
print("Saved deterministic memo ->", WEEKLY_MEMO_PATH)



------------------------------------------------------------------------------
5.1 Build deterministic weekly memo
------------------------------------------------------------------------------


# Weekly GA4 decision memo — week starting 2021-01-25

## 1. Latest Top Issues

1. **Google organic traffic: product view to cart drop-off**
   - Evidence: Current conversion was 11.6% versus a weekly baseline of 17.6%, a gap of 5.9% across 3,444 earlier-step sessions. Estimated lost purchases: 58.1.
   - Trend: Moved up from rank 2 to rank 1.
   - Suggested action: Review product-page clarity, pricing visibility, and add-to-cart prompts for Google Organic Traffic.

2. **Google organic traffic: shipping to payment drop-off**
   - Evidence: Current conversion was 59.9% versus a weekly baseline of 73.6%, a gap of 13.7% across 197 earlier-step sessions. Estimated lost purchases: 19.6.
   - Trend: Moved down from rank 1 to rank 2.
   - Suggested action: Review shipping-to-payment flow, cost transparency, and payment-step messaging for Google Organic Traffic.

3. **Google organic traffic: cart to checkout drop-off**
   - Evidence: Current conversion was 49.1% versus a weekly baseline of 53.4%, a gap of 4.3% across 401 earlier-step sessions. Estimated lost purchases: 9.2.
   - Trend: New in the latest weekly Top Issues list.
   - Suggested action: Review cart-to-checkout friction, delivery messaging, and checkout call-to-action clarity for Google Organic Traffic.

## 2. Model evidence

- The final session-level purchase propensity and prioritisation model is `lightgbm_calibrated`.
- Held-out test PR-AUC is 0.8039 and Brier score is 0.0031.
- The top 1% review list reached 81.1% precision and 69.0% recall.
- The top 100 review list reached 85.0% precision and 27.4% recall.

## 3. Intervention review groups

- **Browse drop-off review:** 50 sessions; median purchase likelihood 0.0%. Diagnostic review list for early product-view drop-off; not a high-likelihood recovery group.
- **Cart drop-off review:** 50 sessions; median purchase likelihood 0.0%. Diagnostic review list for cart-stage friction; not a high-likelihood recovery group.
- **Late checkout recovery:** 100 sessions; median purchase likelihood 71.6%. Model-led recovery list for sessions with much higher purchase likelihood.

## 4. Reporting note

Browse and cart review groups support funnel diagnosis, while late checkout recovery is the strongest model-led recovery group. The Power BI decision summary keeps raw segment fields for filtering, readable labels for reporting, and dynamic trend fields for weekly interpretation.

Saved deterministic memo -> /content/drive/MyDrive/Capstone_Project/reports/weekly_memo_template_20260607.md


# 6.0 Save compact audit

The final working step records what Notebook 04 produced and confirms that the expected Power BI decision output exists. The audit remains compact but includes row counts by `summary_type`, making it easier to verify that Top Issues, model evidence, and intervention group rows were all created.

**Process**:
1. Summarise the reporting run, including weeks covered, latest Top Issues, model name, intervention counts, and runtime.
2. Record the number of decision summary rows by `summary_type` so the Power BI table can be checked quickly.
3. Save a compact audit JSON for reproducibility and report checks.
4. Display the audit so the final notebook can be checked without opening folders manually.


In [9]:
#------------------------------------------------------------------------------
# 6.1 Save compact audit
#------------------------------------------------------------------------------
print_banner("6.1 Save compact audit")

# Calculate run summary values
total_runtime_seconds = round(time.time() - NOTEBOOK_START_TS, 2)
intervention_counts = intervention_summary.set_index("intervention_group")["n_candidates"].to_dict()
decision_rows_by_type = weekly_decision_summary["summary_type"].value_counts().to_dict()

# Store final reporting audit
notebook04_audit = {
    "notebook": "04_reporting_and_decision_summary",
    "created_utc": RUN_TS.isoformat(),
    "window": {"START_DATE": window_start, "END_DATE": window_end},
    "latest_reporting_week": latest_week,
    "weeks_covered": int(funnel_weekly_rates["week_start_date"].nunique()),
    "latest_week_top_issues_count": int(len(latest_week_top3)),
    "final_model_name": final_model_name,
    "final_model_test_pr_auc": model_evidence["test_pr_auc"],
    "combined_intervention_candidates": int(len(combined_candidates)),
    "intervention_counts": {str(k): int(v) for k, v in intervention_counts.items()},
    "weekly_decision_summary_rows": int(len(weekly_decision_summary)),
    "weekly_decision_summary_rows_by_type": {str(k): int(v) for k, v in decision_rows_by_type.items()},
    "weekly_decision_summary_created": WEEKLY_DECISION_SUMMARY_PATH.exists(),
    "weekly_memo_created": WEEKLY_MEMO_PATH.exists(),
    "runtime_seconds": total_runtime_seconds}

# Save audit JSON
save_json_overwrite(NOTEBOOK04_AUDIT_PATH, notebook04_audit)

display(pd.DataFrame([notebook04_audit]))
print("Saved audit ->", NOTEBOOK04_AUDIT_PATH)



------------------------------------------------------------------------------
6.1 Save compact audit
------------------------------------------------------------------------------


,notebook,created_utc,window,latest_reporting_week,weeks_covered,latest_week_top_issues_count,final_model_name,final_model_test_pr_auc,combined_intervention_candidates,intervention_counts,weekly_decision_summary_rows,weekly_decision_summary_rows_by_type,weekly_decision_summary_created,weekly_memo_created,runtime_seconds
0,04_reporting_and_decision_summary,2026-06-07T06:10:22.997714+00:00,"{'START_DATE': '20201101', 'END_DATE': '20210228'}",2021-01-25,14,3,lightgbm_calibrated,0.8039,200,"{'Browse drop-off review': 50, 'Cart drop-off review': 50, 'Late checkout recovery': 100}",49,"{'top_issue': 41, 'model_evidence': 5, 'intervention_group': 3}",True,True,3.4500


Saved audit -> /content/drive/MyDrive/Capstone_Project/outputs/data_quality/notebook04_audit.json
